# 数据集二： 构造三个谓词的数据集

#### 0.0 数据集预处理 -- 删除无效数据
users_with_profession_2文件中的id:ID与users文件中的id:ID的属性一一对应，删除users_with_profession_2中在users文件没有对应数据的元组，也就是求两者的交集，将users_with_profession_2有效数据保留，并按照profession排序，profession内按照profession_probability排序

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 定义文件路径（请根据实际情况修改）
users_file = data_dir +  'users.csv'
profession_file = data_dir + 'users_with_profession_2.csv'
output_file = data_dir + 'users_with_profession_valid.csv'

# 读取文件
df_users = pd.read_csv(users_file)
df_prof = pd.read_csv(profession_file)

# 从 users 文件中获取所有有效的用户 id:ID
valid_ids = set(df_users['id:ID'])

# 筛选出 users_with_profession_2 中 id:ID 在有效集合中的记录（求交集）
df_prof_valid = df_prof[df_prof['id:ID'].isin(valid_ids)]

# 对筛选后的数据先按 profession 排序，再在相同 profession 内按 profession_probability 排序
df_prof_valid = df_prof_valid.sort_values(by=['profession', 'profession_probability'], ascending=False)

# 保存结果到输出文件
df_prof_valid.to_csv(output_file, index=False)

print(f"已删除无对应用户的元组，并按照 profession 及 profession_probability 排序，结果保存在 '{output_file}' 文件中。")


#### 0.1 : 数据集预处理 -- 调整属性顺序
：LABEL,id:ID,badges,banned,bio_trans，bio,posts,comments,user_followers,user_following,username,ucNum,upNum,profession,profession_probability
users_translated文件中的bio是翻译后的用户自我介绍，该文件中的id:ID属性可与users_with_profession_valid文件中的id:ID一一对应，对于users_with_profession_valid每一行数据通过id:ID查找users_translated文件中对应的数据的bio保存到users_with_profession_validd中作为新的属性列bio_trans,之后更换上面的属性列的顺序为下面
:LABEL,id:ID,profession_probability，profession,bio_trans,bio,ucNum,upNum，posts,comments,user_followers,user_following,username,badges,banned

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 定义文件路径（请根据实际情况修改）
translated_file = data_dir + 'users_translated.csv'
profession_valid_file = data_dir + 'users_with_profession_valid.csv'
output_file = data_dir + 'users_valid.csv'

# 读取文件
df_translated = pd.read_csv(translated_file)
df_prof_valid = pd.read_csv(profession_valid_file)

# 从 users_translated 文件中提取 id:ID 和 bio 列，bio 为翻译后的自我介绍，
# 并将 bio 列重命名为 bio_trans
df_trans = df_translated[['id:ID', 'bio']].rename(columns={'bio': 'bio_trans'})

# 通过 id:ID 将翻译后的 bio 合并到 users_with_profession_valid 数据中
# 使用左连接，保证 users_with_profession_valid 的所有记录保留
df_merged = pd.merge(df_prof_valid, df_trans, on='id:ID', how='left')

# 调整列顺序，要求顺序如下：
# :LABEL, id:ID, profession_probability, profession, bio_trans, bio, ucNum, upNum, posts, comments, user_followers, user_following, username, badges, banned
desired_columns = [":LABEL", "id:ID", "profession_probability", "profession", "bio_trans", "bio", "ucNum", "upNum", "posts", "comments", "user_followers", "user_following", "username", "badges", "banned"]

# 重新排列列，注意：确保 df_merged 中包含所有这些列
df_final = df_merged[desired_columns]

# 保存结果到输出文件
df_final.to_csv(output_file, index=False)

print(f"已生成文件 '{output_file}'，其中包含新的 bio_trans 列，并调整了列的顺序。")


#### 0.2 ：数据集预处理 -- 删除profession属性内的子串
users_valid文件中profession属性是一个字符串，删除该字符串内的'This person's occupation is '子串

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取 users_valid 文件（请根据实际路径修改文件名）
df = pd.read_csv(data_dir + 'users_valid.csv')

# 删除 profession 列中 "This person's occupation is " 子串
# df['profession'] = df['profession'].str.replace("This person's occupation is ", "", regex=False)
df['profession'] = df['profession'].str.replace("This person works in ", "", regex=False)

# 保存修改后的数据到新文件
df.to_csv(data_dir + 'users_valid.csv', index=False)

print("已删除 profession 字符串中的指定子串，并保存为 'users_valid_modified.csv'")


#### 0.3： 数据集预处理 -- 根据user的bio属性进行职业分类
我有一个user.csv文件，文件内容如下，bio字段是每个用户的自我简介，有的用户没写自我简介，也就是空的，我想根据用户的自我简介，将用户分类为不同职业：医疗健康职业、公共服务职业、公共管理和政府职业、艺术职业、法律职业、教育职业、商务金融职业、军事职业、体育健身职业，如果一个用户不符合前面所有职业就把他划分为其他职业里，写代码完成上面职业划分（每个职业都用其英文名），我要求使用准确度高的预训练的NLP模型根据bio实现用户职业分类

In [0]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm  # 导入 tqdm

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'

# 加载模型和 tokenizer（从本地加载）
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# 将模型转移到GPU（如果可用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

# 读取 CSV 文件中的数据
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
csv_file = data_dir + 'users_valid_test1_del.csv'
print("CSV file:", csv_file)
df = pd.read_csv(csv_file)
prefix = 'This person works in '

# 获取所有的 "bio" 字段内容
all_bios = df['bio'].tolist()
print(f"数据加载完毕，共有 {len(all_bios)} 个用户")

# 职业分类字典：key 值用于打标签，value 值用于职业分类推理
professions = {
    'Healthcare': prefix + "Healthcare profession (doctor,nurse,dentist,veterinarian,etc.)",   ## 医疗健康职业
    'Government': prefix + "Government profession (politician,official,civil servant,diplomats,etc.)",   ## 政府职业
    'Industrial, agricultural and engineering': prefix + "Industrial, agricultural and engineering occupations (farmers, engineers, construction workers, mechanics, etc.)", ## 工农职业
    'Arts': prefix + "Arts profession (painter,musician,actor,dancer,photographer,writer,sculptor,etc.)",   ## 艺术职业
    'Legal': prefix + "Legal profession (lawyer,judge,etc.)",  ## 法律职业
    'Education': prefix + "Education profession (student,teacher,professor,educational administrator,etc.)",  ## 教育职业
    'Business and finance': prefix + "Business and finance profession (accountant,banker,manager,CEO,founder, entrepreneur,etc.)", ## 商务金融职业
    'Military': prefix + "Military profession (Soldiers, veterans,army, navy, airforce,etc.)",   ## 军事职业
    'Sports and fitness': prefix + "Sports and fitness profession (athlete,coach,fitness trainer,referee,etc.)",  ## 体育健身职业
    'service industry': prefix + "Career in service industry (chef, waiter, Courier, hotel manager, etc.)", ## 服务业
    # 'uncertain': "This person's occupation is uncertain"
}

# 定义推理函数
def check_nli_batch(topic, batch_bios):
    """
    对每一批次的 bio 进行 NLI 推理，判断其是否与给定的职业类别匹配
    """
    # 创建每个输入样本的 "bio + profession" 对
    batch_inputs = [f"{bio} [SEP] {topic}" for bio in batch_bios]

    # 编码批量输入
    encoding = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 模型推理
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 选择 "entailment" 和 "contradiction" 的 logits
        entail_contradiction_logits = logits[:, [0, 2]]

        # 计算 softmax 概率
        probs = entail_contradiction_logits.softmax(dim=1)

        # 获取 "entailment" 类别的概率（标签为真时的概率）
        prob_label_is_true = probs[:, 1].cpu().numpy()
        
        return prob_label_is_true

# 预处理函数：空的 bio 字段分类为 "Other profession"
def preprocess_bio(bio):
    if not isinstance(bio, str) or len(bio.strip()) == 0:
        return "Other profession"  # 如果 bio 不是字符串类型或为空字符串，返回默认分类
    return bio  # 否则返回原始的 bio


# 对所有的 "bio" 字段进行预处理
df['bio'] = df['bio'].apply(preprocess_bio)

# 筛选出 profession 为 'uncertain' 且 bio 不为 '0' 的用户
filtered_df = df[(df['profession'] == 'uncertain') & (df['bio'].apply(lambda x: str(x).strip() not in ['', '0']))]
# 获取所有的 "bio" 字段内容
all_bios = filtered_df['bio'].tolist()
print(f"需要重新分类的用户数量： {len(all_bios)}")


# # 获取处理后的所有 "bio" 字段内容
# all_bios = df['bio'].tolist()
# print(f"数据加载完毕，共有 {len(all_bios)} 个用户")

# 假设每批次处理 32 个文本
batch_size = 32
results = []
probabilities_list = []  # 用于保存职业的概率

# 使用 tqdm 显示进度条
for i in tqdm(range(0, len(all_bios), batch_size), desc="Processing Bios"):
    batch_bios = all_bios[i:i + batch_size]
    # 为每个职业分类进行推理
    profession_probs = []  # 存储每个职业类别对应的概率数组
    profession_keys = []   # 存储每个职业类别的 key 值（标签）
    for key, topic in professions.items():
        probabilities = check_nli_batch(topic, batch_bios)
        profession_probs.append(probabilities)
        profession_keys.append(key)

    # 计算每个样本中各个职业类别的概率，选择最大概率的职业类别
    for j in range(len(batch_bios)):
        # 如果 bio 为 "Other profession"，则不进行推理，直接分类为 "Other profession"
        if batch_bios[j] == "Other profession":
            results.append("Other profession")
            probabilities_list.append(0.0)
        else:
            # 找出最大概率的职业类别对应的索引
            max_prob_idx = max(range(len(profession_probs)), key=lambda x: profession_probs[x][j])
            max_prob = profession_probs[max_prob_idx][j]
            results.append(profession_keys[max_prob_idx])
            probabilities_list.append(max_prob)

# 将结果添加到 DataFrame
# df['profession'] = results
# df['profession_probability'] = probabilities_list
# 将结果添加到原始 DataFrame 的相应行
df.loc[filtered_df.index, 'profession'] = results
df.loc[filtered_df.index, 'profession_probability'] = probabilities_list


# 保存包含新列的 CSV 文件
output_csv_file = data_dir + 'users_valid_test2.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


#### 0.4 数据集预处理 -- 根据概率排序

In [0]:
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
csv_file = data_dir + 'users_valid_test2.csv'
print("CSV file:", csv_file)
df = pd.read_csv(csv_file)
# 假设 df 是读取后的 DataFrame
df_sorted = df.sort_values(by=['profession', 'profession_probability'], ascending=[False, False])

# 保存排序后的 DataFrame 到 CSV 文件
output_sorted_csv = data_dir + 'users_valid_test2.csv'
df_sorted.to_csv(output_sorted_csv, index=False)
print(f"排序后的文件已保存到 {output_sorted_csv}")

#### 0.5 统计 'profession' 列各个取值的数量

In [2]:
import pandas as pd

# 读取 CSV 文件
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
df = pd.read_csv(data_dir + 'users_valid_test2.csv')
# df = pd.read_csv(data_dir + 'users_with_profession_1.csv')

# 统计 'profession' 列各个取值的数量
profession_counts = df['profession'].value_counts()

# 打印结果
print(profession_counts)
print(len(df))

profession
Education                                   3411
Industrial, agricultural and engineering    2225
Military                                    2091
Government                                  1540
uncertain                                   1209
Arts                                        1198
Business and finance                         929
service industry                             375
Sports and fitness                           311
Healthcare                                   212
Legal                                        190
Other profession                              67
Name: count, dtype: int64
13758


#### 0.6 统计users_valid_test1文件中bio为None的元组数量

In [32]:
import pandas as pd

data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取 CSV 文件
df = pd.read_csv(data_dir + 'users_valid_test1.csv')

# 找出 bio 为空或为 '0' 的行，并将 profession 赋值为 "uncertain"
df.loc[df['bio'].apply(lambda x: str(x).strip() in ['', '0']), 'profession'] = 'uncertain'

# 如果需要统计满足条件的记录数量
none_count = df['bio'].apply(lambda x: str(x).strip() in ['', '0']).sum()
print("满足条件的记录数量：", none_count)

# 保存修改后的 CSV 文件
output_csv_file = data_dir + 'users_valid_test1.csv'
df.to_csv(output_csv_file, index=False)
print(f"修改后的结果已保存到 {output_csv_file}")


满足条件的记录数量： 1209
修改后的结果已保存到 /home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/users_valid_test1.csv


#### 0.7将posts文件中workload1_probability>0.11541586的元组的workload1_label赋值为workload1_entailment

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取 posts.csv 文件
posts_df = pd.read_csv(data_dir + 'posts.csv')

# 更新满足条件的行，将 workload1_label 赋值为 "workload1_entailment"
posts_df.loc[posts_df['workload1_probability'] > 0.11541586, 'workload1_label'] = 'workload1_entailment'

# 如需要保存更新后的 DataFrame 到新文件
posts_df.to_csv(data_dir + 'posts.csv', index=False)

print("更新完成！")


profession
Education                                   3411
Industrial, agricultural and engineering    2225
Military                                    2091
Government                                  1540
uncertain                                   1209
Arts                                        1198
Business and finance                         929
service industry                             375
Sports and fitness                           311
Healthcare                                   212
Legal                                        190
Other profession                              67
Name: count, dtype: int64
13758

### 2.验证谓词ABC的独立性 ：计算PA,PB,PC,PAB,PAC,PBC,PABC的概率

#### 2.1 计算 PABC 概率

In [8]:
import pandas as pd
# 计算 PABC的概率： 统计用户评论的 nli_label 为 'bart_entailment' 且帖子的 workload1_label 为 'workload1_entailment'  且用户的职业 profession='Military'的记录数量，以及用户-评论-帖子的总记录数量
# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 定义文件路径（请根据实际情况修改）
users_df  = pd.read_csv(data_dir +'users_valid_test.csv')
posts_df  = pd.read_csv(data_dir + 'posts.csv')
comments_df  = pd.read_csv(data_dir + 'comments.csv')

user_profession = 'Education'

# 关联：comments.creator 与 users.id 关联
merged_df = pd.merge(comments_df, users_df, left_on='creator', right_on='id:ID', suffixes=('_comment', '_user'))

# 再关联：comments.post 与 posts.id 关联
merged_df = pd.merge(merged_df, posts_df, left_on='post', right_on='id:ID', suffixes=('', '_post'))

# 统计所有三元组数量（只统计能够关联上的记录）
total_triples = merged_df.shape[0]
print("所有三元组的数量：", total_triples)

# 统计满足条件的三元组数量
# 条件：用户 profession == 'Arts profession'
#      评论 nli_label == 'bart_entailment'
#      帖子 workload1_label == 'workload1_entailment'

condition_df = merged_df[
    (merged_df['profession'] == user_profession) &
    (merged_df['nli_label'] == 'bart_entailment') &
    (merged_df['oracle_label'] == 'deepseek_yes')
]
condition_triples = condition_df.shape[0]
print("满足条件的三元组数量：", condition_triples)
print('PABC的概率：', condition_triples / total_triples)

所有三元组的数量： 32999
满足条件的三元组数量： 0
PABC的概率： 0.0


#### 2.2 计算 PA PB PC 概率

In [5]:
import pandas as pd
user_profession = 'Military' 

# 读取 CSV 文件
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
comments_df = pd.read_csv(data_dir + 'comments.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')

# 1. 计算 P(A) - 评论的关联帖子满足 M7 的比例
# 获取满足 M7 的帖子 (即 post 中 nli_label == 'entailment')
posts_entailment = posts_df[posts_df['workload1_label'] == 'workload1_entailment']
valid_posts = posts_entailment['id:ID'].values
P_A = len(valid_posts) / len(posts_df)
print('[valid_posts_num]: ', len(valid_posts))
# 2. 计算 P(B) - 评论情感为 "agree" 的比例
P_B = len(comments_df[comments_df['nli_label'] == 'bart_entailment']) / len(comments_df)

print(f"P(A): {P_A}")
print(f"P(B): {P_B}")

# 3.计算PC的概率

df = pd.read_csv(data_dir + 'users_valid_test2.csv') 
# 统计 'profession' 列各个取值的数量
profession_counts = df['profession'].value_counts()
# 计算总数
total_count = len(df)
# 构造字典，保存每种 profession 的名称、数量、所占数据的比例
profession_stats = {
    profession: {
        '数量': count,
        '比例': count / total_count
    }
    for profession, count in profession_counts.items()
}
print("P(C)：" + str(profession_stats[user_profession]['比例']))
# # 输出结果
# for profession, stats in profession_stats.items():
#     print(f"职业: {profession}, 数量: {stats['数量']}, 比例: {stats['比例']:.2%}")
#     


[valid_posts_num]:  387
P(A): 0.04476058292852186
P(B): 0.12438491774639686
P(C)：0.151984300043611


#### 2.3 计算 PAB ,PBC 概率

In [3]:
# 1. 计算PBC
import pandas as pd
# 定义数据目录
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
user_profession = 'Military' 
# 读取 CSV 文件
users_df = pd.read_csv(data_dir + 'users_valid_test2.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')
# 将评论数据与用户数据合并，关联键为用户ID
merged_df = pd.merge(comments_df, users_df, left_on='creator', right_on='id:ID', suffixes=('_comment', '_user'))
total_records = merged_df.shape[0]
print(f"总的用户-评论记录数量：{total_records}")
# 筛选条件：评论的 nli_label 为 'bart_entailment' 且用户的职业为 'Military'
filtered_df = merged_df[(merged_df['nli_label'] == 'bart_entailment') & (merged_df['profession'] == 'Military')]
# 计算满足条件的记录数量
filtered_records = filtered_df.shape[0]
print(f"满足条件的记录数量：{filtered_records}")
print('PBC: ' , filtered_records/total_records)

# 2.计算PAB
merged_df = pd.merge(comments_df, posts_df, left_on='post', right_on='id:ID', suffixes=('_comment', '_post'))
total_comment_post_records = merged_df.shape[0]
condition_df = merged_df[
    (merged_df['nli_label'] == 'bart_entailment') &
    (merged_df['workload1_label'] == 'workload1_entailment')
]
condition_count = condition_df.shape[0]
if total_comment_post_records > 0:
    PAB_probability = condition_count / total_comment_post_records
else:
    PAB_probability = None  # 避免除以零的情况
print(f"满足条件的评论-帖子记录数量：{condition_count}")
print(f"评论-帖子的总记录数量：{total_comment_post_records}")
if PAB_probability is not None:
    print(f"PAB 的概率：{PAB_probability:.4f}")
else:
    print("无法计算 PAB 的概率，因为评论-帖子的总记录数量为零。")


总的用户-评论记录数量：25546
满足条件的记录数量：431
PBC:  0.01687152587489235
满足条件的评论-帖子记录数量：296
评论-帖子的总记录数量：33525
PAB 的概率：0.0088


记录bio列值为0，profession=‘uncertain’的用户，有多少人发布的post的workload1_label= ‘workload1_entailment’ 或者发布的comment的nli_label=‘bart_entailment’ 。
删除那些对pa,pb谓词没有影响的bio =0 的用户

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 CSV 文件
users_df = pd.read_csv(data_dir + 'users_valid_test1.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 预处理：过滤出 bio 为空或为 '0' 且 profession 为 "uncertain" 的用户
filtered_users = users_df[
    users_df['bio'].apply(lambda x: str(x).strip() in ['', '0']) &
    (users_df['profession'] == 'uncertain')
].copy()
print("符合 bio 条件和 profession 为 'uncertain' 的用户数量：", filtered_users.shape[0])

# 为避免合并时冲突，将用户 ID 列重命名为 "user_id"
filtered_users.rename(columns={'id:ID': 'user_id'}, inplace=True)

# 关联 comments 与 filtered_users：使用 comments.creator 与 filtered_users.user_id 关联
merged_comments = pd.merge(
    comments_df, 
    filtered_users, 
    left_on='creator', 
    right_on='user_id', 
    suffixes=('_comment', '_user')
)

# 筛选评论中 nli_label 为 'bart_entailment'
comments_condition = merged_comments[merged_comments['nli_label'] == 'bart_entailment']

# 关联 posts 与 filtered_users：使用 posts.creator 与 filtered_users.user_id 关联
merged_posts = pd.merge(
    posts_df, 
    filtered_users, 
    left_on='creator', 
    right_on='user_id', 
    suffixes=('_post', '_user')
)

# 筛选帖子中 workload1_label 为 'workload1_entailment'
posts_condition = merged_posts[merged_posts['workload1_label'] == 'workload1_entailment']

# 提取符合条件的用户ID（去重）
user_ids_comments = set(comments_condition['user_id'].unique())
user_ids_posts = set(posts_condition['user_id'].unique())

# 取并集得到符合任一条件的用户数
users_meeting_condition = user_ids_comments.union(user_ids_posts)
count_users = len(users_meeting_condition)

print("符合条件的用户数量：", count_users)


In [ ]:
# 计算需要删除的用户集合：在 filtered_users 中但不在 users_meeting_condition 中的用户
user_del = set(filtered_users['user_id'].unique()) - users_meeting_condition
print("需要删除的用户数量：", len(user_del))
# 在原始的 users_df 中，用户ID 存放在 "id:ID" 列中
# 删除这些用户
users_df_updated = users_df[~users_df['id:ID'].isin(user_del)]

# 保存更新后的文件
output_csv_file = data_dir + 'users_valid_test1_del.csv'
users_df_updated.to_csv(output_csv_file, index=False)
print(f"更新后的用户文件已保存到 {output_csv_file}")

Education                                   3411
Industrial, agricultural and engineering    2225
Military                                    2091
Government                                  1540
uncertain                                   1209
Arts                                        1198
Business and finance                         929
service industry                             375
Sports and fitness                           311
Healthcare                                   212
Legal                                        190
Other profession                              67

所有三元组的数量： 24960
满足条件的三元组数量： 26


#### 保存那些发布的comment的nli_label=‘bart_entailment’且comment对应的post的workload1_label= ‘workload1_entailment’ 且的用户的所有信息到新文件user_pad.csv中

In [ ]:
import pandas as pd

# 定义数据目录
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取 CSV 文件
users_df = pd.read_csv(data_dir + 'users_valid_test2.csv')
posts_df = pd.read_csv(data_dir + 'posts.csv')
comments_df = pd.read_csv(data_dir + 'comments.csv')

# 筛选出 nli_label 为 'bart_entailment' 的评论
filtered_comments = comments_df[comments_df['nli_label'] == 'bart_entailment']

# 将筛选后的评论与帖子数据合并，关联字段为评论的 'post' 和帖子的 'id:ID'
merged_comments_posts = pd.merge(
    filtered_comments,
    posts_df,
    left_on='post',
    right_on='id:ID',
    suffixes=('_comment', '_post')
)

# 从合并后的数据中，筛选出帖子的 workload1_label 为 'workload1_entailment' 的记录
filtered_comments_posts = merged_comments_posts[
    merged_comments_posts['workload1_label'] == 'workload1_entailment'
]

# 获取符合条件的用户ID
eligible_user_ids = filtered_comments_posts['creator_comment'].unique()

# 从用户数据中提取符合条件的用户信息
eligible_users = users_df[users_df['id:ID'].isin(eligible_user_ids)]

# 将符合条件的用户信息保存到新的 CSV 文件
output_file = data_dir + 'user_pab.csv'
eligible_users.to_csv(output_file, index=False)

print(f"符合条件的用户信息已保存到 {output_file}")


# 职业分类列表
professions = [
    prefix + "Healthcare profession",   ## 医疗健康职业
    prefix + "Public service profession", ## 公共服务职业
    prefix + "Public administration and government profession", ## 公共管理和政府职业
    prefix + "Arts profession",   #· 艺术职业
    prefix + "Legal profession",  ## 法律职业
    prefix + "Education profession",  ## 教育职业
    prefix + "Business and finance profession", ## 商务金融职业
    prefix + "Military profession",   # 军事职业
    prefix + "Sports and fitness profession" ,  ## 体育健身职业
    "This person's occupation is uncertain"
]

In [ ]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# ========================================
# 加载数据
# ========================================
try:
    users = pd.read_csv(data_dir + 'users_valid.csv')
    posts = pd.read_csv(data_dir + 'posts.csv')
    comments = pd.read_csv(data_dir + 'comments.csv')
except FileNotFoundError as e:
    print(f"错误：文件未找到 - {e}")
    exit(1)

# ========================================
# 合并数据（使用INNER JOIN确保完整三元组）
# ========================================
try:
    # 第1步：评论 ↔ 用户
    merged_users = pd.merge(
        comments,
        users,
        left_on='creator',
        right_on='id:ID',    # 根据实际列名调整
        how='inner',
        suffixes=('_comment', '_user')
    )
    
    # 第2步：结果 ↔ 帖子 (修复位置：补全括号)
    final_merged = pd.merge(
        merged_users,
        posts,
        left_on='post',
        right_on='id:ID',   # 根据实际列名调整
        how='inner',
        suffixes=('', '_post'))  # 这里补上缺失的括号
    
except KeyError as e:  # 现在语法正确
    print(f"合并错误：请检查列名是否正确 - {e}")
    exit(1)

# ========================================
# 统计总数
# ========================================
total_triples = len(final_merged)
print(f"\n总三元组数量 (用户-评论-帖子): {total_triples:,}")

# ========================================
# 筛选条件（根据实际列名调整）
# ========================================
try:
    filtered = final_merged[
        # 用户职业条件
        (final_merged['profession'] == 'Public administration and government profession') &  # 假设用户表列名为profession
        # 评论标签条件
        (final_merged['nli_label'] == 'bart_entailment') &    # 假设评论表列名为nli_label
        # 帖子标签条件
        (final_merged['workload1_label'] == 'workload1_entailment')  # 假设帖子表列名为workload1_label
    ]
except KeyError as e:
    print(f"筛选错误：请检查条件列名是否存在 - {e}")
    exit(1)

# ========================================
# 输出结果
# ========================================
print(f"满足条件的三元组数量: {len(filtered):,}\n")

##### 替代: 使用管道推理文本，速度极其慢

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

# 本地模型路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'

# 创建分类pipeline
classifier = pipeline(
    "zero-shot-classification",
    model=model_dir,  # 使用本地模型
    device=0 if torch.cuda.is_available() else -1,  # 自动使用GPU
    framework="pt"
)

# 定义职业分类配置（按优先级顺序）
categories = {
    "healthcare": "medical, doctor, nurse, hospital, healthcare",
    "public_service": "public service, community service, social work",
    "public_administration": "government, public administration, civil service",
    "arts": "artist, music, painting, theater, film, design",
    "legal": "lawyer, legal, attorney, court, judge",
    "education": "teacher, professor, education, school, university",
    "business_finance": "business, finance, CEO, founder, entrepreneur, bank",
    "military": "military, army, navy, airforce, soldier,veteran",
    "sports_fitness": "sports, athlete, fitness, gym, coach"
}

data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
csv_file = data_dir + 'users_valid.csv'
# 读取数据
df = pd.read_csv(csv_file)
df['bio'] = df['bio'].fillna('')  # 处理空值

# 分类处理函数
def classify_bio(text):
    if not text.strip():  # 跳过空简介
        return "other", 0.0
    
    # 执行零样本分类
    result = classifier(
        text,
        candidate_labels=list(categories.keys()),
        hypothesis_template="This person works in {}.",
        multi_label=False  # 单标签分类
    )
    
    # 获取最高概率的类别和对应的概率
    top_category = result['labels'][0]
    top_score = result['scores'][0]
    
    # 检查最高概率是否超过阈值
    if top_score > 0.1:
        return top_category, top_score
    return "other", 1-top_score

# 使用tqdm显示进度条
tqdm.pandas(desc="Classifying Bios")
df[['category', 'category_probability']] = df['bio'].progress_apply(lambda x: pd.Series(classify_bio(x)))

# 保存结果
output_csv_file = data_dir + 'users_valid_cat.csv'
df.to_csv(output_csv_file, index=False)
print(f"分类结果已保存至: {output_csv_file}")
